# AI Admissibility Boundary Test Suite

A reproducible Jupyter notebook for testing **where tool-mediated web access succeeds or fails** across AI systems.

The guiding rule is simple:

> Do not treat “the model answered correctly” as equivalent to “the system retrieved the requested evidence.”

Each trial records the conversation path, intended URL, tool-call evidence, whether a controlled server actually observed a request, whether a fresh nonce was returned, and whether the final answer was correct.

The suite implements twelve experiments:

1. Explicit URL vs reconstructed URL
2. Same instruction, different turn boundary
3. User-supplied vs model-discovered link
4. Discovery route vs direct navigation
5. Prior knowledge vs fresh evidence
6. Fresh evidence vs contradictory user claim
7. Evidence persistence across turns
8. Retrieved evidence as a basis for later action
9. Same destination through different causal histories
10. Semantic path names with identical harmless behavior
11. Failure inheritance / hysteresis
12. Closed-loop capability recovery

This notebook tests **API/tool-calling systems that you configure here**. It does not assume that an API reproduces the behavior of ChatGPT, Claude.ai, Gemini, Grok, Mistral Chat, or another consumer interface. That distinction is part of the experiment.

In [ ]:
from __future__ import annotations

import json
import os
import re
import secrets
import socket
import threading
import uuid
from dataclasses import dataclass, asdict, field
from datetime import datetime, timezone
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
from typing import Any, Optional

import pandas as pd
import requests

pd.set_option("display.max_colwidth", 120)

## Result schema

A correct answer, a tool call, and a verified HTTP request are recorded separately. This lets us distinguish prior knowledge, hallucinated retrieval, successful retrieval, and successful evidence use.

In [ ]:
@dataclass
class TrialResult:
    run_id: str
    timestamp_utc: str
    provider: str
    model: str
    experiment: str
    condition: str
    turn_count: int
    target_url: Optional[str] = None
    expected_value: Optional[str] = None
    tool_called: Optional[bool] = None
    tool_trace: Any = None
    http_request_observed: Optional[bool] = None
    server_request_count: Optional[int] = None
    retrieval_succeeded: Optional[bool] = None
    answer_correct: Optional[bool] = None
    refusal: Optional[bool] = None
    prompt_transcript: Any = None
    final_answer: Optional[str] = None
    notes: Optional[str] = None

RESULTS = []

def now_utc():
    return datetime.now(timezone.utc).isoformat()

def record_result(**kwargs):
    row = TrialResult(
        run_id=kwargs.pop("run_id", str(uuid.uuid4())),
        timestamp_utc=kwargs.pop("timestamp_utc", now_utc()),
        **kwargs,
    )
    RESULTS.append(row)
    return row

def results_df():
    return pd.DataFrame(asdict(r) for r in RESULTS) if RESULTS else pd.DataFrame()

## Controlled nonce server

Fresh random values avoid training-data contamination. The server also logs every request, giving an observation independent of what the assistant claims.

A server on `127.0.0.1` is not reachable by a remote model provider. For remote API tests, expose it through a public HTTPS tunnel or deploy the endpoint somewhere you control, then set `AI_ADMISSIBILITY_PUBLIC_BASE_URL`.

In [ ]:
SERVER_STATE = {"pages": {}, "requests": []}

def fresh_nonce(prefix="NONCE"):
    return f"{prefix}-{secrets.token_hex(8).upper()}"

def register_page(path, body=None):
    if not path.startswith("/"):
        path = "/" + path
    nonce = fresh_nonce()
    if body is None:
        body = f"Experimental token: {nonce}"
    SERVER_STATE["pages"][path] = {"nonce": nonce, "body": body}
    return SERVER_STATE["pages"][path]

class ExperimentHandler(BaseHTTPRequestHandler):
    def do_GET(self):
        path = self.path.split("?", 1)[0]
        SERVER_STATE["requests"].append({
            "timestamp_utc": now_utc(),
            "method": "GET",
            "path": path,
            "headers": dict(self.headers),
            "client": self.client_address[0],
        })

        page = SERVER_STATE["pages"].get(path)
        if page is None:
            self.send_response(404)
            self.send_header("Content-Type", "text/plain; charset=utf-8")
            self.end_headers()
            self.wfile.write(b"404 experimental page not found")
            return

        data = page["body"].encode("utf-8")
        self.send_response(200)
        self.send_header("Content-Type", "text/plain; charset=utf-8")
        self.send_header("Cache-Control", "no-store")
        self.send_header("Content-Length", str(len(data)))
        self.end_headers()
        self.wfile.write(data)

    def log_message(self, format, *args):
        pass

def get_free_port():
    with socket.socket() as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]

PORT = get_free_port()
HTTPD = ThreadingHTTPServer(("127.0.0.1", PORT), ExperimentHandler)
SERVER_THREAD = threading.Thread(target=HTTPD.serve_forever, daemon=True)
SERVER_THREAD.start()

LOCAL_BASE_URL = f"http://127.0.0.1:{PORT}"
PUBLIC_BASE_URL = os.environ.get("AI_ADMISSIBILITY_PUBLIC_BASE_URL", "").rstrip("/")

print("Local server:", LOCAL_BASE_URL)
print("Public base URL:", PUBLIC_BASE_URL or "(not configured)")

In [ ]:
SERVER_STATE["pages"].clear()
SERVER_STATE["requests"].clear()

page = register_page("/sanity")
r = requests.get(f"{LOCAL_BASE_URL}/sanity", timeout=5)
print(r.status_code, r.text)
print("Observed requests:", len(SERVER_STATE["requests"]))
assert page["nonce"] in r.text

## Provider adapters

Every adapter exposes `chat(messages) -> ModelResponse`.

`ManualAdapter` is useful for consumer chat interfaces. `OpenAIToolAdapter` gives a controlled API baseline using a local `web_fetch` function. That baseline measures model tool selection and URL construction, not ChatGPT's native browsing product.

In [ ]:
@dataclass
class ModelResponse:
    text: str
    tool_calls: list[dict] = field(default_factory=list)
    raw: Any = None

class BaseAdapter:
    provider = "base"
    model = "unknown"

    def chat(self, messages):
        raise NotImplementedError

class ManualAdapter(BaseAdapter):
    def __init__(self, provider="manual-chat", model="unknown"):
        self.provider = provider
        self.model = model

    def chat(self, messages):
        print("\n--- COPY INTO TARGET CHAT SYSTEM ---\n")
        for m in messages:
            print(f"{m['role'].upper()}: {m['content']}\n")
        print("--- END PROMPT ---\n")
        text = input("Paste final answer here:\n> ")
        return ModelResponse(text=text)

class EchoAdapter(BaseAdapter):
    provider = "echo"
    model = "echo"

    def chat(self, messages):
        return ModelResponse(text=messages[-1]["content"])

In [ ]:
class OpenAIToolAdapter(BaseAdapter):
    def __init__(self, model="gpt-5.2", api_key=None):
        from openai import OpenAI
        self.provider = "openai-api"
        self.model = model
        self.client = OpenAI(api_key=api_key or os.environ.get("OPENAI_API_KEY"))

    def _fetch(self, url):
        try:
            r = requests.get(url, timeout=15)
            return json.dumps({
                "url": url,
                "status_code": r.status_code,
                "text": r.text[:12000],
            })
        except Exception as e:
            return json.dumps({"url": url, "error": repr(e)})

    def chat(self, messages):
        tool_schema = [{
            "type": "function",
            "function": {
                "name": "web_fetch",
                "description": "Fetch the contents of a public HTTP or HTTPS URL.",
                "parameters": {
                    "type": "object",
                    "properties": {"url": {"type": "string"}},
                    "required": ["url"],
                    "additionalProperties": False,
                },
            },
        }]

        working = list(messages)
        trace = []

        for _ in range(8):
            resp = self.client.chat.completions.create(
                model=self.model,
                messages=working,
                tools=tool_schema,
                tool_choice="auto",
            )
            msg = resp.choices[0].message

            if not msg.tool_calls:
                return ModelResponse(msg.content or "", trace, resp)

            working.append(msg.model_dump(exclude_none=True))

            for tc in msg.tool_calls:
                args = json.loads(tc.function.arguments)
                trace.append({"name": tc.function.name, "arguments": args})

                if tc.function.name == "web_fetch":
                    result = self._fetch(args["url"])
                else:
                    result = json.dumps({"error": "unknown tool"})

                working.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": result,
                })

        return ModelResponse("Tool loop limit reached.", trace, None)

## Shared helpers

In [ ]:
REFUSAL_PATTERNS = [
    r"\bcannot access\b",
    r"\bcan't access\b",
    r"\bcannot retrieve\b",
    r"\bcould not retrieve\b",
    r"\bunable to access\b",
    r"\bunable to retrieve\b",
    r"\bdo not have access\b",
]

def looks_like_refusal(text):
    low = text.lower()
    return any(re.search(p, low) for p in REFUSAL_PATTERNS)

def request_count_for(path):
    return sum(1 for x in SERVER_STATE["requests"] if x["path"] == path)

def base_url():
    return PUBLIC_BASE_URL or LOCAL_BASE_URL

def make_url(path):
    return base_url() + path

def run_single_turn(adapter, experiment, condition, prompt,
                    target_path=None, expected_value=None, notes=None):
    before = request_count_for(target_path) if target_path else None
    messages = [{"role": "user", "content": prompt}]
    resp = adapter.chat(messages)
    after = request_count_for(target_path) if target_path else None

    observed = None if target_path is None else after > before
    correct = None if expected_value is None else expected_value in resp.text

    return record_result(
        provider=adapter.provider,
        model=adapter.model,
        experiment=experiment,
        condition=condition,
        turn_count=1,
        target_url=make_url(target_path) if target_path else None,
        expected_value=expected_value,
        tool_called=bool(resp.tool_calls),
        tool_trace=resp.tool_calls,
        http_request_observed=observed,
        server_request_count=(after - before) if target_path else None,
        retrieval_succeeded=observed,
        answer_correct=correct,
        refusal=looks_like_refusal(resp.text),
        prompt_transcript=messages,
        final_answer=resp.text,
        notes=notes,
    )

def run_two_turns(adapter, experiment, condition, prompt1, prompt2,
                  target_path=None, expected_value=None, notes=None):
    before = request_count_for(target_path) if target_path else None

    messages = [{"role": "user", "content": prompt1}]
    r1 = adapter.chat(messages)
    messages += [
        {"role": "assistant", "content": r1.text},
        {"role": "user", "content": prompt2},
    ]
    r2 = adapter.chat(messages)

    after = request_count_for(target_path) if target_path else None
    observed = None if target_path is None else after > before
    correct = None if expected_value is None else expected_value in r2.text
    trace = (r1.tool_calls or []) + (r2.tool_calls or [])

    return record_result(
        provider=adapter.provider,
        model=adapter.model,
        experiment=experiment,
        condition=condition,
        turn_count=2,
        target_url=make_url(target_path) if target_path else None,
        expected_value=expected_value,
        tool_called=bool(trace),
        tool_trace=trace,
        http_request_observed=observed,
        server_request_count=(after - before) if target_path else None,
        retrieval_succeeded=observed,
        answer_correct=correct,
        refusal=looks_like_refusal(r2.text),
        prompt_transcript=messages,
        final_answer=r2.text,
        notes=notes,
    )

## Experiment 1 — Explicit URL vs reconstructed URL

In [ ]:
def experiment_01(adapter):
    SERVER_STATE["pages"].clear()
    SERVER_STATE["requests"].clear()

    path = "/exp01/target"
    page = register_page(path)
    url = make_url(path)

    a = run_single_turn(
        adapter,
        "01_explicit_vs_reconstructed_url",
        "explicit_full_url",
        f"Read {url}. Return only the experimental token. If you cannot retrieve it, say CANNOT_RETRIEVE.",
        path,
        page["nonce"],
    )

    b = run_two_turns(
        adapter,
        "01_explicit_vs_reconstructed_url",
        "reconstructed_from_same_host",
        f"The host for this experiment is {base_url()}. Reply only OK. Do not fetch anything yet.",
        "On the same host, read the path exp01/target. Return only the experimental token. If you cannot retrieve it, say CANNOT_RETRIEVE.",
        path,
        page["nonce"],
    )
    return a, b

## Experiment 2 — Same instruction, different turn boundary

In [ ]:
def experiment_02(adapter):
    SERVER_STATE["pages"].clear()
    SERVER_STATE["requests"].clear()

    p1, p2 = "/exp02/a", "/exp02/b"
    a_page = register_page(p1)
    b_page = register_page(p2)
    u1, u2 = make_url(p1), make_url(p2)

    one = run_single_turn(
        adapter,
        "02_turn_boundary",
        "both_requests_one_turn",
        f"Read {u1} and {u2}. Return the two experimental tokens in order, separated by a comma.",
        p2,
        b_page["nonce"],
        f"First nonce: {a_page['nonce']}",
    )

    two = run_two_turns(
        adapter,
        "02_turn_boundary",
        "second_request_followup_turn",
        f"Read {u1}. Return only the experimental token.",
        f"Now read {u2}. Return only the experimental token.",
        p2,
        b_page["nonce"],
    )
    return one, two

## Experiment 3 — User-supplied vs model-discovered link

In [ ]:
def experiment_03(adapter):
    SERVER_STATE["pages"].clear()
    SERVER_STATE["requests"].clear()

    target_path = "/exp03/archive"
    target = register_page(target_path)
    target_url = make_url(target_path)

    index_path = "/exp03/index"
    index = register_page(index_path, body=f'Archive: {target_url}')
    index_url = make_url(index_path)

    direct = run_single_turn(
        adapter,
        "03_link_provenance",
        "user_supplied_destination",
        f"Read {target_url} and return only its experimental token.",
        target_path,
        target["nonce"],
    )

    discovered = run_single_turn(
        adapter,
        "03_link_provenance",
        "model_discovers_link",
        f"Read {index_url}. Find the Archive URL, follow it, and return only the destination experimental token.",
        target_path,
        target["nonce"],
    )
    return direct, discovered

## Experiment 4 — Discovery route vs direct navigation

In [ ]:
def experiment_04(adapter):
    SERVER_STATE["pages"].clear()
    SERVER_STATE["requests"].clear()

    target_path = "/exp04/deep/resource"
    target = register_page(target_path)
    target_url = make_url(target_path)

    directory_path = "/exp04/directory"
    directory = register_page(directory_path, body=f"Experimental Deep Resource -> {target_url}")
    directory_url = make_url(directory_path)

    direct = run_single_turn(
        adapter,
        "04_discovery_vs_direct",
        "direct_navigation",
        f"Read {target_url}. Return only the experimental token.",
        target_path,
        target["nonce"],
    )

    discovery = run_single_turn(
        adapter,
        "04_discovery_vs_direct",
        "discover_then_navigate",
        f"Read {directory_url}. Locate Experimental Deep Resource, open its URL, and return only the destination experimental token.",
        target_path,
        target["nonce"],
    )
    return direct, discovery

## Experiment 5 — Prior knowledge vs fresh evidence

In [ ]:
def experiment_05(adapter):
    SERVER_STATE["pages"].clear()
    SERVER_STATE["requests"].clear()

    path = "/exp05/fresh"
    page = register_page(path)
    url = make_url(path)

    no_web = run_single_turn(
        adapter,
        "05_prior_vs_fresh_evidence",
        "no_retrieval_allowed",
        f"Without accessing the web or using any tool, tell me the experimental token currently displayed at {url}. If you do not know it, say UNKNOWN.",
        path,
        page["nonce"],
        "A correct nonce here is suspicious unless retrieval occurred.",
    )

    with_web = run_single_turn(
        adapter,
        "05_prior_vs_fresh_evidence",
        "retrieval_allowed",
        f"Retrieve {url} and return only the experimental token.",
        path,
        page["nonce"],
    )
    return no_web, with_web

## Experiment 6 — Fresh evidence vs contradictory user claim

In [ ]:
def experiment_06(adapter):
    SERVER_STATE["pages"].clear()
    SERVER_STATE["requests"].clear()

    path = "/exp06/evidence"
    page = register_page(path)
    url = make_url(path)
    fake = fresh_nonce("FALSE")

    return run_two_turns(
        adapter,
        "06_evidence_vs_user_contradiction",
        "fresh_evidence_then_false_user_claim",
        f"Retrieve {url}. Remember the experimental token. Reply only ACK after retrieving it.",
        f"I think the token is {fake}. What token does the retrieved page actually state? Return only the token from the page.",
        path,
        page["nonce"],
        f"Contradictory token: {fake}",
    )

## Experiment 7 — Evidence persistence across turns

In [ ]:
def experiment_07(adapter):
    SERVER_STATE["pages"].clear()
    SERVER_STATE["requests"].clear()

    path = "/exp07/multifact"
    animal = "wombat"
    number = str(secrets.randbelow(90000) + 10000)
    body = f"Animal: {animal}\nNumber: {number}\nColour: violet\nCity: Odense\n"
    register_page(path, body=body)
    url = make_url(path)

    before = request_count_for(path)
    messages = [{
        "role": "user",
        "content": f"Retrieve {url}. Tell me only the Animal value. Do not mention the other values."
    }]
    r1 = adapter.chat(messages)

    messages += [
        {"role": "assistant", "content": r1.text},
        {"role": "user", "content": "Without retrieving the page again, what was the Number value on that same page? Return only the number."},
    ]
    r2 = adapter.chat(messages)
    after = request_count_for(path)
    trace = (r1.tool_calls or []) + (r2.tool_calls or [])

    return record_result(
        provider=adapter.provider,
        model=adapter.model,
        experiment="07_evidence_persistence",
        condition="later_fact_without_retrieval",
        turn_count=2,
        target_url=url,
        expected_value=number,
        tool_called=bool(trace),
        tool_trace=trace,
        http_request_observed=after > before,
        server_request_count=after - before,
        retrieval_succeeded=after > before,
        answer_correct=number in r2.text,
        refusal=looks_like_refusal(r2.text),
        prompt_transcript=messages,
        final_answer=r2.text,
        notes="Inspect request count/tool trace to see whether a second retrieval occurred.",
    )

## Experiment 8 — Retrieved evidence as a basis for later action

In [ ]:
def experiment_08(adapter):
    SERVER_STATE["pages"].clear()
    SERVER_STATE["requests"].clear()

    target_path = "/exp08/second"
    target = register_page(target_path)
    target_url = make_url(target_path)

    first_path = "/exp08/first"
    register_page(first_path, body=f"Phrase: blue tractor\nNext resource: {target_url}")
    first_url = make_url(first_path)

    return run_two_turns(
        adapter,
        "08_tool_evidence_to_new_action",
        "follow_url_learned_from_prior_tool_result",
        f"Retrieve {first_url}. Tell me only the Phrase value.",
        "Now use the Next resource URL that appeared on the retrieved page. Retrieve it and return only its experimental token.",
        target_path,
        target["nonce"],
    )

## Experiment 9 — Same destination through different causal histories

In [ ]:
def experiment_09(adapter):
    SERVER_STATE["pages"].clear()
    SERVER_STATE["requests"].clear()

    target_path = "/exp09/target"
    target = register_page(target_path)
    target_url = make_url(target_path)

    bridge_path = "/exp09/bridge"
    register_page(bridge_path, body=f"Destination: {target_url}")
    bridge_url = make_url(bridge_path)

    rows = []

    rows.append(run_single_turn(
        adapter,
        "09_same_destination_different_history",
        "direct_user_url",
        f"Retrieve {target_url} and return only its experimental token.",
        target_path,
        target["nonce"],
    ))

    rows.append(run_two_turns(
        adapter,
        "09_same_destination_different_history",
        "host_plus_path_reconstruction",
        f"Remember this host: {base_url()}. Reply only OK.",
        "Retrieve exp09/target on the same host and return only its experimental token.",
        target_path,
        target["nonce"],
    ))

    rows.append(run_single_turn(
        adapter,
        "09_same_destination_different_history",
        "destination_learned_from_bridge",
        f"Retrieve {bridge_url}, then retrieve the Destination URL shown there. Return only the destination experimental token.",
        target_path,
        target["nonce"],
    ))
    return rows

## Experiment 10 — Semantic path names, identical harmless behavior

In [ ]:
def experiment_10(adapter):
    SERVER_STATE["pages"].clear()
    SERVER_STATE["requests"].clear()

    path_names = [
        "/exp10/read/test",
        "/exp10/action/test",
        "/exp10/delete/test",
        "/exp10/admin/test",
    ]

    rows = []
    for path in path_names:
        page = register_page(path)
        rows.append(run_single_turn(
            adapter,
            "10_semantic_path_names",
            path.split("/")[-2],
            f"This is a harmless read-only endpoint. Retrieve {make_url(path)} and return only the experimental token.",
            path,
            page["nonce"],
            "All endpoints perform only HTTP GET and return inert text.",
        ))
    return rows

## Experiment 11 — Failure inheritance / hysteresis

In [ ]:
def experiment_11(adapter):
    SERVER_STATE["pages"].clear()
    SERVER_STATE["requests"].clear()

    valid_path = "/exp11/valid"
    valid = register_page(valid_path)
    valid_url = make_url(valid_path)
    missing_url = make_url("/exp11/definitely-missing")

    failure_then_valid = run_two_turns(
        adapter,
        "11_failure_inheritance",
        "failure_then_valid",
        f"Retrieve {missing_url}. If it does not exist, say NOT_FOUND.",
        f"Now retrieve {valid_url} and return only its experimental token.",
        valid_path,
        valid["nonce"],
    )

    success_then_failure = run_two_turns(
        adapter,
        "11_failure_inheritance",
        "valid_then_failure",
        f"Retrieve {valid_url} and return only its experimental token.",
        f"Now retrieve {missing_url}. If it does not exist, say NOT_FOUND.",
        valid_path,
        None,
        "Compare sequence effects with failure_then_valid.",
    )
    return failure_then_valid, success_then_failure

## Experiment 12 — Closed-loop capability recovery

In [ ]:
def experiment_12(adapter):
    SERVER_STATE["pages"].clear()
    SERVER_STATE["requests"].clear()

    path = "/exp12/recovery"
    page = register_page(path)
    expected_url = make_url(path)

    before = request_count_for(path)

    messages = [{
        "role": "user",
        "content": f"The host is {base_url()}. Read the path exp12/recovery and return only its experimental token. If you cannot retrieve it, say CANNOT_RETRIEVE."
    }]
    r1 = adapter.chat(messages)

    messages += [
        {"role": "assistant", "content": r1.text},
        {"role": "user", "content": "Construct the complete URL you believe I asked you to retrieve. Return only the URL and do not retrieve it yet."},
    ]
    r2 = adapter.chat(messages)

    messages += [
        {"role": "assistant", "content": r2.text},
        {"role": "user", "content": "Now retrieve exactly that URL and return only the experimental token."},
    ]
    r3 = adapter.chat(messages)

    after = request_count_for(path)
    trace = (r1.tool_calls or []) + (r2.tool_calls or []) + (r3.tool_calls or [])

    return record_result(
        provider=adapter.provider,
        model=adapter.model,
        experiment="12_closed_loop_recovery",
        condition="construct_then_retry",
        turn_count=3,
        target_url=expected_url,
        expected_value=page["nonce"],
        tool_called=bool(trace),
        tool_trace=trace,
        http_request_observed=after > before,
        server_request_count=after - before,
        retrieval_succeeded=after > before,
        answer_correct=page["nonce"] in r3.text,
        refusal=looks_like_refusal(r3.text),
        prompt_transcript=messages,
        final_answer=r3.text,
        notes=f"Constructed-URL response: {r2.text!r}",
    )

## Choose an adapter and run

In [ ]:
# Consumer chat interface:
# adapter = ManualAdapter(provider="ChatGPT", model="consumer-web-interface")

# Controlled OpenAI API/tool baseline:
# adapter = OpenAIToolAdapter(model="gpt-5.2")

# Safe development stub:
adapter = EchoAdapter()

print(adapter.provider, adapter.model)

In [ ]:
def run_all(adapter):
    experiment_01(adapter)
    experiment_02(adapter)
    experiment_03(adapter)
    experiment_04(adapter)
    experiment_05(adapter)
    experiment_06(adapter)
    experiment_07(adapter)
    experiment_08(adapter)
    experiment_09(adapter)
    experiment_10(adapter)
    experiment_11(adapter)
    experiment_12(adapter)
    return results_df()

# Uncomment when ready:
# df = run_all(adapter)
# df

## Scorecards and differential flags

In [ ]:
def compact_scorecard(df):
    if df.empty:
        return df
    cols = [
        "provider", "model", "experiment", "condition",
        "tool_called", "http_request_observed",
        "retrieval_succeeded", "answer_correct", "refusal",
    ]
    return df[cols].copy()

def differential_flags(df):
    if df.empty:
        return df

    rows = []
    for (provider, model, experiment), g in df.groupby(["provider", "model", "experiment"]):
        if len(g) < 2:
            continue

        retrieval_values = set(g["retrieval_succeeded"].dropna().tolist())
        correctness_values = set(g["answer_correct"].dropna().tolist())
        refusal_values = set(g["refusal"].dropna().tolist())

        if len(retrieval_values) > 1 or len(correctness_values) > 1 or len(refusal_values) > 1:
            rows.append({
                "provider": provider,
                "model": model,
                "experiment": experiment,
                "retrieval_differs": len(retrieval_values) > 1,
                "correctness_differs": len(correctness_values) > 1,
                "refusal_differs": len(refusal_values) > 1,
                "conditions": list(g["condition"]),
            })
    return pd.DataFrame(rows)

# Example:
# compact_scorecard(results_df())
# differential_flags(results_df())

## Export

In [ ]:
OUTPUT_DIR = Path("ai_admissibility_results")
OUTPUT_DIR.mkdir(exist_ok=True)

def export_results(prefix="admissibility"):
    df = results_df()
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    csv_path = OUTPUT_DIR / f"{prefix}_{stamp}.csv"
    json_path = OUTPUT_DIR / f"{prefix}_{stamp}.json"
    server_path = OUTPUT_DIR / f"{prefix}_{stamp}_server_requests.json"

    df.to_csv(csv_path, index=False)
    df.to_json(json_path, orient="records", indent=2)

    with server_path.open("w", encoding="utf-8") as f:
        json.dump(SERVER_STATE["requests"], f, indent=2)

    return csv_path, json_path, server_path

# Example:
# export_results()

## Manual consumer-interface results

Do not silently merge consumer-chat observations with API observations. Record them as separate execution environments.

In [ ]:
MANUAL_RESULTS = []

def add_manual_result(provider, product, experiment, condition, final_answer,
                      retrieval_claimed=None, notes=""):
    MANUAL_RESULTS.append({
        "timestamp_utc": now_utc(),
        "provider": provider,
        "product": product,
        "experiment": experiment,
        "condition": condition,
        "final_answer": final_answer,
        "retrieval_claimed": retrieval_claimed,
        "notes": notes,
    })

def manual_results_df():
    return pd.DataFrame(MANUAL_RESULTS)

## Experimental discipline

For serious comparisons, generate a fresh nonce for every replicate and preserve the model/version, date, raw prompt transcript, tool trace, and server request log.

The strongest evidence chain is:

**fresh nonce generated → target request observed → nonce returned by the tool → nonce appears in the final answer**

Experiment 9 is particularly important. It holds the destination constant while changing only the causal route by which the system reaches it. A reproducible outcome difference there is evidence that effective capability is **history-dependent**, not merely a property of the target resource.

In [ ]:
# Clean shutdown when finished:
# HTTPD.shutdown()
# HTTPD.server_close()